# 7.21 未來答案該對到哪個位置？


沿用「〈B〉／雨／停／了／〈E〉」：距離 2 的分支讀到「停」時，要學的是〈E〉；讀到「了」時，序列裡已經沒有往後數第 2 個 token。若為了湊出一個答案，拿下一篇文章的開頭當答案，就把兩份無關的材料接成了同一句。

因此每條分支都要有自己的目標位置與有效範圍。這個例子的規則是：**可以預測 EOS 本身，但不能越過它**；沒有目標的位置填 [7.3](https://birdhackor.github.io/tiny-perceptron-vlm/7.3.html) 的 −100，不計代價。角色邊界、PAD 與拼接在一起的多篇文件，也要按資料規則另外處理，不能把整批資料的答案直接整排平移，讓超出尾端的部分繞回開頭。

下面只計算兩條分支的交叉熵，不建立或訓練完整的模型。字表次序固定為雨、停、了、〈E〉，ID 分別是 0、1、2、3（〈B〉從來不是目標，這裡省略）。人工分數全設為 0，每個候選的 softmax 機率都是四分之一；先用這個容易手算的情況，核對哪些位置真的計分。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
import torch.nn.functional as F

vocab = ["雨", "停", "了", "〈E〉"]
next_labels = torch.tensor([0, 1, 2, 3])
later_labels = torch.cat([next_labels[1:], torch.tensor([-100])])
next_scores = torch.zeros(4, 4)
later_scores = torch.zeros(4, 4)

next_loss = F.cross_entropy(next_scores, next_labels, ignore_index=-100)
later_loss = F.cross_entropy(later_scores, later_labels, ignore_index=-100)
weight = 0.5
total = next_loss + weight * later_loss
print("距離2的目標", later_labels.tolist())
print("各分支有效目標數", (next_labels != -100).sum().item(), (later_labels != -100).sum().item())
print("各分支平均代價", round(next_loss.item(), 4), round(later_loss.item(), 4))
print("加上輔助代價後", round(total.item(), 4))

每張分數表有四列（四個已讀位置）、四欄（字表裡的候選）。距離 1 的四個目標依序是雨、停、了、〈E〉。距離 2 的目標，就是去掉距離 1 目標的第一項、在最後補一個 −100：`next_labels[1:]` 去掉第一項，`torch.cat` 把它和 `[-100]` 接成一條，得到 `[1,2,3,-100]`，也就是停、了、〈E〉和不計分。`cross_entropy` 在每條分支內只對有效目標取平均，所以有效目標數印出 4 和 3。每個有效目標的代價都是 `−log(1/4)`，兩條分支的平均都約 1.3863；總代價是距離 1 的代價加上半份距離 2 的代價，約 1.3863＋0.5×1.3863≈2.0794。主要目標仍是下一個 token，距離 2 這份額外加上的代價叫**輔助代價**（auxiliary loss）。

這裡選擇「距離 1 的平均代價＋0.5×距離 2 的平均代價」，是為了把係數和分母說清楚；係數 0.5 是本例指定的，不是所有 MTP 方法的預設。兩條分支先各自除以自己的有效目標數，再相加。若改成直接加總，距離 2 只有 3 項，總和約 4.159，自然比距離 1 的 4 項總和約 5.545 小，看起來像距離 2 猜得比較準，其實只是少算一項。真正訓練時，用總代價的梯度同時更新共用的核心與兩條分支，兩份目標都會影響學習；這段程式只做固定分數的數值核對。

如果兩篇文章被裝進同一列，還要保存它們的邊界。對較遠的目標而言，不只終點要有效，從當前位置走到終點的路上也不能跨過文件結束。對話微調則要明確決定哪些助手回答的位置受監督；原本距離 1 的遮罩不能不加思考就複製給每個距離。

練習只把 `weight` 改成 0。先預測：有效目標數仍是 4 與 3，兩條分支的平均代價仍約 1.3863，總代價只剩距離 1 的 1.3863；再執行核對。若真的用這個總代價更新，距離 2 的代價不再提供梯度：算出第二份數字，和用它教模型，是兩件事。

<details>
<summary>選讀：目標移位與論文配方</summary>

若目前位置是 t，距離 1 的目標在 t+1，距離 2 的在 t+2。這是 [7.4](https://birdhackor.github.io/tiny-perceptron-vlm/7.4.html) 下一位置對齊向不同距離的延伸，不是改變因果注意力。Gloeckle 等人的[第 2 節](https://arxiv.org/pdf/2404.19737v1)加總不同距離的交叉熵；DeepSeek-V3 的[第 2.2 節式 24–25](https://arxiv.org/pdf/2412.19437v2)則定義各順序模組的代價與係數。比較配方時要讀它們完整的約定，不把本節按有效目標取平均、係數 0.5 的做法搬過去當成相同的公式。

</details>
